# 01. Adult Census Income - 탐색적 데이터 분석(EDA)

- 작성: SKALA 광주 캠퍼스 4반 2조 / 박영서
- 목적: `src/` 모듈을 재사용해 원본 데이터를 살펴보고 전처리 전후를 비교한다.
- 주의: 검증이 끝난 함수는 노트북이 아니라 `src/` 모듈에 두고, 노트북은 탐색 용도로만 사용한다.

In [ ]:
# 프로젝트 루트를 import 경로에 추가 (노트북은 notebooks/ 하위에서 실행됨)
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Image

from src import clean, stats_analysis, viz

pd.set_option("display.max_columns", 50)

## 1. 데이터 로딩 — Pandas vs Polars

In [ ]:
raw_path = clean.download_raw()  # 캐시가 있으면 재사용

pdf, t_pandas = clean.load_with_pandas(raw_path)
pldf, t_polars = clean.load_with_polars(raw_path)

clean.compare_loaders(pdf, pldf, t_pandas, t_polars)

In [ ]:
pdf.head()

## 2. 기본 EDA — 구조 · 결측치 · 타깃 분포

In [ ]:
pdf.info()

In [ ]:
eda = clean.basic_eda(pdf)

print("결측치(건수, 비율%):", eda["null_counts"])
print("중복 행:", eda["n_duplicates"])
print("타깃 분포:", eda["target_dist"])
eda["describe"]

## 3. 전처리 — 중복 제거 + 결측치 대체

In [ ]:
df, history = clean.clean_data(pdf)

print("전처리 이력:", history)
print("전처리 후 결측치 합계:", int(df.isna().sum().sum()))
print("전처리 후 shape:", df.shape)

## 4. 시각화 — Seaborn 정적 차트 / Plotly 인터랙티브 차트

In [ ]:
static_path = viz.plot_static_eda(df)
interactive_path = viz.plot_interactive(df)


Image(filename=str(static_path))

## 5. 통계 분석 — 기술통계 · 상관 · t-test · 카이제곱

In [ ]:
display(stats_analysis.describe_numeric(df))

corr = stats_analysis.correlation_matrix(df)
display(corr)

print("상관 상위 5쌍:", stats_analysis.top_correlations(corr))

In [ ]:
ttest = stats_analysis.run_ttest(df, value_col="hours-per-week")
chi2 = stats_analysis.run_chi2(df, col_a="sex", col_b="income")

print(ttest["interpretation"])
print(chi2["interpretation"])

## 6. 다음 단계

모델 학습 · 저장 · 리포트 생성까지 포함한 전체 흐름은 아래 명령으로 한 번에 실행한다.

```bash
python -m src.run_pipeline
```